# 006 - QUBO Analysis  

## 1.0 -- Admin



###  1.01 -- Python Libraries 

In [1]:
import pandas as pd 
import Quantum_Optimization_Functions as qof 
import environment as envir 

import matplotlib.pyplot as plt 
import seaborn as sns 
from tabulate import tabulate 

import numpy as np
from scipy.optimize import minimize, differential_evolution
from itertools import product

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

### 1.02 -- Graphics Adjustment 

In [2]:
plt.style.use("seaborn-v0_8")
print_flag = False

### 1.03 -- Environment Status

In [3]:
data, headers = envir.environment_state()
print(tabulate(data, headers=headers, tablefmt="grid"))

+-------------------------+-----------+
| System/Library          | Version   |
+=========================+===========+
| Python                  | 3.10.9    |
+-------------------------+-----------+
| numpy                   | 2.2.6     |
+-------------------------+-----------+
| torch                   | 2.9.1     |
+-------------------------+-----------+
| matplotlib              | 3.10.8    |
+-------------------------+-----------+
| seaborn                 | 0.13.2    |
+-------------------------+-----------+
| qiskit                  | 1.4.4     |
+-------------------------+-----------+
| qiskit_machine_learning | 0.8.4     |
+-------------------------+-----------+
| yfinance                | 1.2.0     |
+-------------------------+-----------+


## 2.0 -- Data 

### 2.01 -- Import data 

In [4]:
target_tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'JPM', 'XOM', 'JNJ', 'V']
summary_df, daily_returns = qof.get_assets(companies_list=target_tickers)


### 2.02 -- Create a covariance Matrix 

In [5]:
corr = daily_returns.corr()
Covariance_Matrix = daily_returns.cov()
assets = summary_df['Ticker'].to_list()

In [6]:
if print_flag: 
    plt.figure(figsize=(8,6))
    sns.heatmap(corr, annot=True, cmap='coolwarm', xticklabels=assets, yticklabels=assets)
    plt.title("Correlation Matrix of Portfolio Assets")
    plt.show()

In [7]:
Returns_Matrix = daily_returns.to_numpy()
Expected_Return_Vector = qof.compute_expected_return_vector(Returns_Matrix) 

### 2.03 -- Data Summary 

In [8]:
N_Assets = Covariance_Matrix.shape[0]
print("Number of assets:", N_Assets)
print("Covariance_Matrix shape:", Covariance_Matrix.shape)
print("Expected_Return_Vector shape:", Expected_Return_Vector.shape)

Number of assets: 10
Covariance_Matrix shape: (10, 10)
Expected_Return_Vector shape: (10,)


## 3.0 -- Model 

### 3.01 -- QUBO Parameters 

In [9]:
# Trade-off between risk and return
Risk_Aversion_Parameter = 0.5  # tune this as you like

# You can also store metadata for later comparison with QAOA
QUBO_Metadata = {
    "Risk_Aversion_Parameter": Risk_Aversion_Parameter,
    "Description": "Mean-variance portfolio QUBO without explicit constraints"
}

### 3.02 -- Build Q Matrix 

In [10]:
Q_Matrix = qof.build_Q_Matrix(
    Covariance_Matrix=Covariance_Matrix,
    Expected_Return_Vector=Expected_Return_Vector,
    Risk_Aversion_Parameter=Risk_Aversion_Parameter
)

print("Q_Matrix shape:", Q_Matrix.shape)


Q_Matrix shape: (10, 10)


### 3.03 -- Fetch Portfolios  

In [11]:
all_portfolios = qof.generate_portfolios(target_tickers)
if print_flag: 
    print(len(all_portfolios))
    print(all_portfolios[:5]) 

### 3.10 -- Calculate Scores 

In [12]:
Results_List = []

for Binary_Decision_Vector in generate_all_binary_portfolios(N_Assets=N_Assets):
    QUBO_Energy = compute_QUBO_Energy(
        Binary_Decision_Vector=Binary_Decision_Vector,
        Q_Matrix=Q_Matrix
    )
    Portfolio_Risk, Portfolio_Return = compute_portfolio_risk_and_return(
        Binary_Decision_Vector=Binary_Decision_Vector,
        Covariance_Matrix=Covariance_Matrix,
        Expected_Return_Vector=Expected_Return_Vector
    )

    Results_List.append({
        "Binary_Decision_Vector": Binary_Decision_Vector,
        "QUBO_Energy": QUBO_Energy,
        "Portfolio_Risk": Portfolio_Risk,
        "Portfolio_Return": Portfolio_Return
    })

Results_DataFrame = pd.DataFrame(Results_List)
print("Number of evaluated portfolios:", len(Results_DataFrame))
Results_DataFrame.head()

NameError: name 'generate_all_binary_portfolios' is not defined